# Azure File Storage(Azure Files) 연결/CRUD 테스트

`.env`를 로드해서 Azure Files(파일 공유)에 연결 → 디렉터리/파일 생성(Create) → 조회(Read/List) → 수정(Update) → 삭제(Delete)까지 한 번씩 해보는 노트북입니다.

`blob_storage_test.ipynb`와 목적은 같지만, Blob과 File은 완전히 다른 서비스라서 SDK도 `azure-storage-file-share`로 다르고, 진짜 폴더 구조를 씁니다.

## 사전 준비
1. 아래 패키지가 설치되어 있어야 합니다.

```bash
pip install azure-storage-file-share python-dotenv
```
2. 프로젝트 루트(`azure-doc-ai-service/`)의 `.env`에 아래 값이 채워져 있어야 합니다 (없다면 `.env.example`을 복사).

```
AZURE_FILE_STORAGE_CONNECTION_STRING=<연결 문자열>
AZURE_FILE_SHARE_NAME=documents
```

파일 공유가 Blob과 다른 스토리지 계정에 있다면, `AZURE_FILE_STORAGE_CONNECTION_STRING`에 **그 파일 공유가 있는 계정**의 연결 문자열을 넣으면 됩니다.

In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

from azure.core.exceptions import ResourceExistsError, ResourceNotFoundError
from azure.storage.fileshare import ShareServiceClient
from dotenv import load_dotenv

# 이 노트북(azure-doc-ai-service/notebooks/)의 부모 폴더(azure-doc-ai-service/)에 있는 .env를 로드
ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH)

CONNECTION_STRING = os.environ["AZURE_FILE_STORAGE_CONNECTION_STRING"]
SHARE_NAME = os.environ.get("AZURE_FILE_SHARE_NAME", "documents")

# CRUD 테스트에 사용할 디렉터리/파일 이름 (File은 진짜 폴더 구조라서 디렉터리를 먼저 만들어야 함)
DIRECTORY_NAME = "test"
FILE_NAME = "hello.txt"
FILE_PATH = f"{DIRECTORY_NAME}/{FILE_NAME}"

print("SHARE_NAME:", SHARE_NAME)
print("FILE_PATH:", FILE_PATH)

## 1. 클라이언트 연결

연결 문자열로 `ShareServiceClient`를 만들고, 실제로 인증/네트워크가 되는지 확인하기 위해 계정에 있는 파일 공유 목록을 한 번 조회합니다.

In [ ]:
service_client = ShareServiceClient.from_connection_string(CONNECTION_STRING)

print("연결 성공, 계정 내 파일 공유 목록(최대 10개):")
for i, share in enumerate(service_client.list_shares()):
    if i >= 10:
        print("  ...")
        break
    print(f"  - {share.name}")

## 2. Create - 공유/디렉터리 생성 + 파일 업로드

`SHARE_NAME` 공유가 없으면 새로 만들고(이미 있으면 그대로 사용), `DIRECTORY_NAME` 디렉터리를 만든 뒤, 그 안에 텍스트 파일을 업로드합니다.

Blob과 달리 File API는 `overwrite` 옵션이 없고 `create_file`을 호출하면 기존 파일이 있어도 그냥 덮어써버립니다. 그래서 "진짜 새로 생성됐는지"를 확인하려고 업로드 전에 `exists()`로 직접 체크합니다.

In [ ]:
share_client = service_client.get_share_client(SHARE_NAME)

try:
    share_client.create_share()
    print(f"공유 '{SHARE_NAME}' 생성됨")
except ResourceExistsError:
    print(f"공유 '{SHARE_NAME}' 이미 존재함 (그대로 사용)")

try:
    share_client.create_directory(DIRECTORY_NAME)
    print(f"디렉터리 '{DIRECTORY_NAME}' 생성됨")
except ResourceExistsError:
    print(f"디렉터리 '{DIRECTORY_NAME}' 이미 존재함 (그대로 사용)")


def create_file(file_path: str, content: str) -> None:
    """file_path가 이미 있으면 에러를 내도록 만들어서 진짜 '생성'만 테스트한다 (File API는 overwrite 옵션이 없어서 직접 체크)."""
    file_client = share_client.get_file_client(file_path)
    if file_client.exists():
        raise ResourceExistsError(f"'{file_path}'가 이미 존재합니다.")
    data = content.encode("utf-8")
    file_client.upload_file(data)
    print(f"'{file_path}' 생성 완료 ({len(content)}자)")


initial_content = f"hello from file_storage_test.ipynb (created_at={datetime.now(timezone.utc).isoformat()})"

try:
    create_file(FILE_PATH, initial_content)
except ResourceExistsError:
    print(f"'{FILE_PATH}'가 이미 존재합니다. 삭제 후 다시 실행하거나 FILE_NAME을 바꾸세요.")

## 3. Read - 목록 조회 + 다운로드

디렉터리 안의 파일 목록을 조회하고, 방금 만든 `FILE_PATH`의 내용을 다운로드해서 확인합니다.

In [ ]:
def list_files(directory_name: str = DIRECTORY_NAME) -> list[str]:
    directory_client = share_client.get_directory_client(directory_name)
    return [item.name for item in directory_client.list_directories_and_files()]


def read_file(file_path: str) -> str:
    file_client = share_client.get_file_client(file_path)
    data = file_client.download_file().readall()
    return data.decode("utf-8")


print(f"디렉터리 '{DIRECTORY_NAME}' 안의 파일 목록:")
for name in list_files():
    print(f"  - {name}")

print(f"\n'{FILE_PATH}' 내용:")
print(read_file(FILE_PATH))

## 4. Update - 같은 파일 덮어쓰기

새 내용으로 같은 파일을 다시 업로드해서 내용이 바뀌는지 확인합니다. (File API는 `create_file`/`upload_file`이 곧 덮어쓰기라서 Blob처럼 `overwrite=True` 옵션이 따로 없습니다.)

In [ ]:
def update_file(file_path: str, content: str) -> None:
    file_client = share_client.get_file_client(file_path)
    file_client.upload_file(content.encode("utf-8"))
    print(f"'{file_path}' 갱신 완료 ({len(content)}자)")


updated_content = f"updated content (updated_at={datetime.now(timezone.utc).isoformat()})"
update_file(FILE_PATH, updated_content)

print("\n갱신 후 내용 재조회:")
print(read_file(FILE_PATH))

## 5. Delete - 파일 삭제 (+ 선택: 디렉터리/공유 삭제)

테스트에 쓴 파일을 지웁니다. 디렉터리/공유 자체를 지우는 셀은 기본적으로 실행 안 되게 주석 처리해뒀습니다 (다른 데이터가 든 공유를 실수로 통째로 지우는 걸 막기 위함) — 정말 지우고 싶을 때만 주석을 풀고 실행하세요.

In [ ]:
def delete_file(file_path: str) -> None:
    file_client = share_client.get_file_client(file_path)
    try:
        file_client.delete_file()
        print(f"'{file_path}' 삭제 완료")
    except ResourceNotFoundError:
        print(f"'{file_path}'가 이미 없습니다.")


delete_file(FILE_PATH)

print("\n삭제 후 파일 목록:")
for name in list_files():
    print(f"  - {name}")

In [ ]:
# 디렉터리/공유 자체를 삭제하려면 아래 주석을 풀고 실행하세요 (되돌릴 수 없습니다).
# share_client.delete_directory(DIRECTORY_NAME)
# print(f"디렉터리 '{DIRECTORY_NAME}' 삭제 완료")

# share_client.delete_share()
# print(f"공유 '{SHARE_NAME}' 삭제 완료")